# 01 — Olist Data-Quality Audit

This notebook is the **presentation layer** for the source-data audit. All discovery, validation, and report generation run in reusable Python modules through:

```powershell
python scripts\run_data_quality_audit.py
```

The notebook reads the generated reports; it does not modify or re-audit the raw CSV files.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Could not locate the project root containing src/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import REPORTS_DIR

AUDIT_REPORT_DIR = REPORTS_DIR / 'data_quality'
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## Load generated audit reports

The validation below provides a clear instruction if the command-line pipeline has not been run yet.

In [ ]:
required_reports = {
    'overview': AUDIT_REPORT_DIR / 'table_overview.csv',
    'columns': AUDIT_REPORT_DIR / 'column_quality.csv',
    'timestamps': AUDIT_REPORT_DIR / 'timestamp_issues.csv',
    'issues': AUDIT_REPORT_DIR / 'issue_register.csv',
    'schemas': AUDIT_REPORT_DIR / 'schema_checks.csv',
    'keys': AUDIT_REPORT_DIR / 'key_checks.csv',
    'relationships': AUDIT_REPORT_DIR / 'relationship_checks.csv',
    'business_rules': AUDIT_REPORT_DIR / 'business_rule_checks.csv',
    'summary': AUDIT_REPORT_DIR / 'audit_summary.json',
}
missing_reports = [path for path in required_reports.values() if not path.is_file()]
if missing_reports:
    raise FileNotFoundError(
        'Audit reports are missing. From the repository root, run: '
        'python scripts\\run_data_quality_audit.py'
    )

overview = pd.read_csv(required_reports['overview'])
column_quality = pd.read_csv(required_reports['columns'])
timestamp_issues = pd.read_csv(required_reports['timestamps'])
issue_register = pd.read_csv(required_reports['issues'])
schema_checks = pd.read_csv(required_reports['schemas'])
key_checks = pd.read_csv(required_reports['keys'])
relationship_checks = pd.read_csv(required_reports['relationships'])
business_rule_checks = pd.read_csv(required_reports['business_rules'])
with required_reports['summary'].open(encoding='utf-8') as summary_file:
    audit_summary = json.load(summary_file)

print(
    f"Audit generated: {audit_summary['generated_at_utc']} | "
    f"Files audited: {audit_summary['audited_file_count']} | "
    f"Load errors: {audit_summary['load_error_count']}"
)

## Table overview

This is a structural comparison across source tables. Row counts are table-specific and should not be added together as a business KPI.

In [ ]:
display(overview.sort_values('rows', ascending=False).reset_index(drop=True))

## Missingness by column

Only columns with at least one missing value are shown. Missing values require business interpretation before any cleaning decision.

In [ ]:
columns_with_missing = (
    column_quality.loc[column_quality['missing_values'].gt(0)]
    .sort_values(['missing_percent', 'file_name'], ascending=[False, True])
    .reset_index(drop=True)
)
if columns_with_missing.empty:
    print('No missing values were detected.')
else:
    display(columns_with_missing)

## Duplicate rows and candidate keys

Candidate keys are heuristic suggestions, not database constraints. The database-design phase must confirm table grain and composite keys.

In [ ]:
display(
    overview[[
        'file_name', 'duplicate_rows', 'likely_primary_keys', 'quality_warnings'
    ]].sort_values(['duplicate_rows', 'file_name'], ascending=[False, True])
)

## Timestamp consistency

Timestamp-like text columns are parsed non-destructively. Invalid examples remain in the report for investigation.

In [ ]:
if timestamp_issues.empty:
    print('No populated timestamp-like text columns were detected.')
else:
    display(timestamp_issues.sort_values(['invalid_values', 'file_name'], ascending=[False, True]))

## Potential data-quality problems

These are investigation prompts generated from observed conditions. They are not automatic instructions to delete, impute, or transform data.

In [ ]:
if issue_register.empty:
    print('No potential issues were flagged by the automated checks.')
else:
    display(issue_register.sort_values(['file_name', 'issue_type']).reset_index(drop=True))

## Relational contract checks

These reports validate source schemas, declared single or composite keys, and foreign-key coverage. A `review` result indicates an observed exception requiring a documented decision; it does not automatically justify deleting data.

In [ ]:
print('Schema checks')
display(schema_checks)
print('Declared key checks')
display(key_checks)
print('Foreign-key coverage')
display(relationship_checks)

## Business-rule checks

Controlled order statuses and chronological timestamp expectations are evaluated from the current source files.

In [ ]:
display(
    business_rule_checks.sort_values(
        ['status', 'violation_rows', 'rule_name'],
        ascending=[True, False, True],
    ).reset_index(drop=True)
)

## Next phase

Use this audit to document table grain, expected composite keys, relationship cardinality, structural missingness, and defensible cleaning rules. Cleaning should produce new files in `data/processed`; the source CSVs remain immutable.